# Multi-Class Chest X-Ray Disease Detection
Detects multiple conditions from chest X-rays.

**Folder structure expected:**

Add or remove subfolders to change which diseases are detected — Keras auto-detects class names from folder names.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os
import json


In [ ]:
# Define paths
base_dir = './data'
train_dir = os.path.join(base_dir, 'train')
val_dir   = os.path.join(base_dir, 'val')
test_dir  = os.path.join(base_dir, 'test')

# Auto-detect class names from folder names
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)
print(f"Detected {num_classes} classes: {class_names}")


In [ ]:
# Data generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen  = ImageDataGenerator(rescale=1.0/255)
test_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=32,
    class_mode='categorical'   # multi-class
)
val_generator = val_datagen.flow_from_directory(
    val_dir, target_size=(150, 150), batch_size=32,
    class_mode='categorical'
)
test_generator = test_datagen.flow_from_directory(
    test_dir, target_size=(150, 150), batch_size=32,
    class_mode='categorical'
)

# Save class index mapping for use in server.py
with open('class_indices.json', 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)
print("Saved class_indices.json:", train_generator.class_indices)


In [ ]:
# Build multi-class CNN
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(150, 150, 3)),
    tf.keras.layers.Conv2D(32, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation="softmax")  # one output per class
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",   # changed from binary_crossentropy
    metrics=["accuracy"]
)
model.summary()


In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=15,
    validation_data=val_generator,
    validation_steps=val_generator.samples // val_generator.batch_size,
)


In [ ]:
model.save('disease_model.h5')
print('Model saved as disease_model.h5')


In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.2f}")


In [ ]:
# Plot accuracy/loss curves
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Test a single image
from tensorflow.keras.preprocessing import image as keras_image
import json

with open('class_indices.json') as f:
    class_idx = json.load(f)
idx_to_class = {v: k for k, v in class_idx.items()}

img_path = 'data/test/PNEUMONIA/person1_virus_6.jpeg'  # change this
img = keras_image.load_img(img_path, target_size=(150,150))
img_array = keras_image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

preds = model.predict(img_array)[0]
top_idx = np.argmax(preds)
print(f"Predicted: {idx_to_class[top_idx]}  ({preds[top_idx]*100:.1f}% confidence)")
print("All scores:", {idx_to_class[i]: f"{p*100:.1f}%" for i, p in enumerate(preds)})
